In [1]:
import os
import shutil
import cv2
import numpy as np
from tqdm.notebook import tqdm
import re

# Путь к исходному датасету (где лежат inaction, move, work)
ORIGINAL_DATASET = '/home/shizm/DL_LABs/DL-lab6/data'

# Путь для нового датасета с 8 лучшими кадрами
NEW_DATASET = '/home/shizm/DL_LABs/DL-lab6/data_top8'

# Количество кадров, которые хотим оставить
N_FRAMES = 8

In [2]:
def numerical_sort_key(filename):
    nums = re.findall(r'\d+', filename)
    return int(''.join(nums)) if nums else filename

In [5]:
def compute_frame_importance(frame_paths):
    """
    Принимает список путей ко всем кадрам трека (отсортированный).
    Возвращает индексы кадров, которые нужно оставить (всего N_FRAMES).
    Стратегия: 
      - всегда берём первый и последний кадр.
      - остальные (N_FRAMES-2) выбираем как кадры с максимальной 
        средней абсолютной разностью с предыдущим кадром.
    Все изображения приводятся к размеру 224x224 (как в ConvNeXt).
    """
    if len(frame_paths) <= N_FRAMES:
        return list(range(len(frame_paths)))
    
    # Читаем все изображения, приводим к единому размеру
    imgs = []
    for path in frame_paths:
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            imgs.append(None)
            continue
        img = cv2.resize(img, (224, 224))  # фиксированный размер
        imgs.append(img)
    
    # Вычисляем разности между последовательными кадрами
    diffs = [0]  # для первого кадра разность 0
    for i in range(1, len(imgs)):
        if imgs[i] is None or imgs[i-1] is None:
            diffs.append(0)
            continue
        diff = np.mean(np.abs(imgs[i].astype(np.float32) - imgs[i-1].astype(np.float32)))
        diffs.append(diff)
    
    L = len(frame_paths)
    candidates = list(range(1, L-1))  # исключаем первый и последний
    candidates.sort(key=lambda i: diffs[i], reverse=True)
    selected = [0] + candidates[:N_FRAMES-2] + [L-1]
    selected.sort()
    return selected

In [6]:
# Создаём новый датасет (очищаем, если существует)
if os.path.exists(NEW_DATASET):
    shutil.rmtree(NEW_DATASET)
os.makedirs(NEW_DATASET)

# Для каждого класса
for cls in ['inaction', 'move', 'work']:
    cls_src = os.path.join(ORIGINAL_DATASET, cls)
    cls_dst = os.path.join(NEW_DATASET, cls)
    os.makedirs(cls_dst, exist_ok=True)
    
    # Проходим по всем подпапкам (трекам)
    for root, dirs, files in os.walk(cls_src):
        if root == cls_src:
            continue
        # Определяем имя папки (трека)
        rel_path = os.path.relpath(root, cls_src)
        dst_path = os.path.join(cls_dst, rel_path)
        os.makedirs(dst_path, exist_ok=True)
        
        # Получаем все изображения
        img_files = [f for f in files if f.lower().endswith(('.png','.jpg','.jpeg'))]
        if not img_files:
            continue
        img_files.sort(key=numerical_sort_key)
        full_paths = [os.path.join(root, f) for f in img_files]
        
        # Выбираем индексы самых полезных кадров
        selected_indices = compute_frame_importance(full_paths)
        selected_paths = [full_paths[i] for i in selected_indices]
        
        # Копируем выбранные кадры в новую папку, переименовывая последовательно
        for idx, src_path in enumerate(selected_paths, 1):
            ext = os.path.splitext(src_path)[1]
            dst_name = f"{idx}{ext}"
            dst_full = os.path.join(dst_path, dst_name)
            shutil.copy2(src_path, dst_full)
    
    print(f"Класс {cls} обработан")

Класс inaction обработан
Класс move обработан
Класс work обработан


In [7]:
# Посчитаем, сколько треков и сколько кадров в каждом
for cls in ['inaction', 'move', 'work']:
    cls_path = os.path.join(NEW_DATASET, cls)
    count = 0
    for root, dirs, files in os.walk(cls_path):
        if root == cls_path:
            continue
        img_count = len([f for f in files if f.lower().endswith(('.png','.jpg','.jpeg'))])
        if img_count != 8:
            print(f"ВНИМАНИЕ: {root} содержит {img_count} кадров")
        count += 1
    print(f"{cls}: {count} треков")

inaction: 93 треков
move: 150 треков
work: 342 треков
